# NB15 Decentralized exchanges and AMMs — Chapter 15

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB15-1: constant-product pool — swaps (with fee), adding and removing liquidity, the slippage curve (for study)
import math

class ConstantProductPool:
    """x·y = k pool. fee is a fraction charged on the amount put in (default 0.003 = 30 bp). Shares are sqrt(x·y) at first, proportional after that."""
    def __init__(self, x: float, y: float, fee: float = 0.003):
        self.x, self.y, self.fee = float(x), float(y), fee
        self.shares = {"LP0": math.sqrt(x * y)}            # the first provider's share = geometric mean (v2 equation (13))
    @property
    def k(self): return self.x * self.y
    @property
    def price_x_in_y(self): return self.y / self.x           # marginal price: the price of one X in Y (v2 equation (1))
    def quote_x_for_y(self, dx: float) -> float:              # put in dx of X: how much Y comes back (keeps the product after taking the fee)
        eff = dx * (1 - self.fee)
        return self.y * eff / (self.x + eff)
    def swap_x_for_y(self, dx: float) -> float:
        dy = self.quote_x_for_y(dx)
        self.x += dx; self.y -= dy                             # the fee portion stays in the pool and k grows
        return dy
    def swap_y_for_x(self, dy: float) -> float:
        eff = dy * (1 - self.fee)
        dx = self.x * eff / (self.y + eff)
        self.y += dy; self.x -= dx
        return dx
    def add_liquidity(self, who: str, dx: float) -> float:     # put in X dx and Y dx·(y/x) at the current ratio (v2 equation (12))
        dy = dx * self.y / self.x
        minted = dx / self.x * sum(self.shares.values())
        self.x += dx; self.y += dy
        self.shares[who] = self.shares.get(who, 0) + minted
        return minted
    def remove_liquidity(self, who: str):                      # returns both tokens in proportion to the share, at the ratio of that moment
        s_total = sum(self.shares.values())
        frac = self.shares.pop(who) / s_total
        out_x, out_y = self.x * frac, self.y * frac
        self.x -= out_x; self.y -= out_y
        return out_x, out_y

pool = ConstantProductPool(1000, 1000)
print(f"start: x={pool.x:.1f} y={pool.y:.1f} k={pool.k:,.0f} marginal price (price of one X in Y)={pool.price_x_in_y:.4f}")
print(f"{'order Δx':>8} {'received Δy':>12} {'exec price Δy/Δx':>17} {'slippage':>9} {'marginal price after':>21}")
for dx in [1, 10, 100, 500, 1000]:
    p = ConstantProductPool(1000, 1000)
    before = p.price_x_in_y
    dy = p.swap_x_for_y(dx)
    exec_price = dy / dx
    print(f"{dx:>8} {dy:>12.3f} {exec_price:>17.4f} {1 - exec_price / before:>9.2%} {p.price_x_in_y:>21.4f}")

# where did the fee go — look at k after putting in 100
p = ConstantProductPool(1000, 1000); dy = p.swap_x_for_y(100)
print(f"\nafter putting in 100: Δy received={dy:.3f} (90.909 without a fee), pool x={p.x:.1f} y={p.y:.3f}, k={p.k:,.1f} (1,000,000 before the trade — the 0.3 X fee stayed in the pool and k grew)")

start: x=1000.0 y=1000.0 k=1,000,000 marginal price (price of one X in Y)=1.0000
order Δx  received Δy  exec price Δy/Δx  slippage  marginal price after
       1        0.996            0.9960     0.40%                0.9980
      10        9.872            0.9872     1.28%                0.9803
     100       90.661            0.9066     9.34%                0.8267
     500      332.666            0.6653    33.47%                0.4449
    1000      499.249            0.4992    50.08%                0.2504

after putting in 100: Δy received=90.661 (90.909 without a fee), pool x=1100.0 y=909.339, k=1,000,272.8 (1,000,000 before the trade — the 0.3 X fee stayed in the pool and k grew)


In [ ]:
# NB15-2: the two pans of the scale — impermanent loss by price multiple r, and how far sqrt(k) grown by fees makes up for it (for study)
import math

def impermanent_loss(r: float) -> float:                     # this book's derivation: V_pool / V_hold − 1 = 2√r/(1+r) − 1
    return 2 * math.sqrt(r) / (1 + r) - 1

print("price multiple r and impermanent loss (assuming fee 0 and full correction of the ratio by arbitrage):")
print(f"{'r':>6} {'value if kept (Y)':>18} {'value in pool (Y)':>18} {'impermanent loss':>17}")
x0 = y0 = 1000.0
for r in [0.25, 0.5, 0.8, 1.0, 1.25, 2.0, 4.0]:
    hold = x0 * r * (y0 / x0) + y0                              # X valued at the new price r·(y/x), plus Y
    pool_val = 2 * y0 * math.sqrt(r)
    print(f"{r:>6.2f} {hold:>18.1f} {pool_val:>18.1f} {impermanent_loss(r):>17.2%}")

# the fee pan of the scale — trades go back and forth so the ratio returns to where it was, but k grows. Count the growth of sqrt(k) by trading volume (as a multiple of the pool).
print("\nfee accrual: swaps of size 10, alternating X→Y and Y→X, N times (the ratio stays almost in place, k grows)")
print(f"{'round-trip volume/pool':>22} {'sqrt(k) growth (= fee accrual per share)':>40} {'r where V_pool = V_hold (formula under assumptions)':>51}")
for n_pairs in [10, 100, 1000]:
    p = ConstantProductPool(1000, 1000)
    k0 = p.k
    for _ in range(n_pairs):
        p.swap_x_for_y(10); p.swap_y_for_x(10)
    growth = math.sqrt(p.k / k0) - 1                           # g = √(k2/k1) − 1. Related to f = 1 − √k1/√k2 in v2 equation (4) by f = g/(1+g) (the reference points differ)
    # the r at which fee growth g exactly offsets impermanent loss: (1+g)·2√r/(1+r) = 1 → the two roots of a quadratic
    a = 1 + growth
    disc = (2 * a) ** 2 - 4
    s1, s2 = (2 * a - math.sqrt(disc)) / 2, (2 * a + math.sqrt(disc)) / 2   # the two roots for √r
    print(f"{2 * n_pairs * 10 / 1000:>21.1f}x {growth:>40.4%} {f'{s1**2:.3f} ≤ r ≤ {s2**2:.3f}':>51}")

price multiple r and impermanent loss (assuming fee 0 and full correction of the ratio by arbitrage):
     r  value if kept (Y)  value in pool (Y)  impermanent loss
  0.25             1250.0             1000.0           -20.00%
  0.50             1500.0             1414.2            -5.72%
  0.80             1800.0             1788.9            -0.62%
  1.00             2000.0             2000.0             0.00%
  1.25             2250.0             2236.1            -0.62%
  2.00             3000.0             2828.4            -5.72%
  4.00             5000.0             4000.0           -20.00%

fee accrual: swaps of size 10, alternating X→Y and Y→X, N times (the ratio stays almost in place, k grows)
round-trip volume/pool sqrt(k) growth (= fee accrual per share) r where V_pool = V_hold (formula under assumptions)
                  0.2x                                  0.0299%                                   0.952 ≤ r ≤ 1.050
                  2.0x                                